# Production data cleaning


## 1. Load the raw datasets

In [1]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("../data/cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
marketing = pd.read_csv("../data/marketing_spend.csv")

cleaning_records = []


## 2. Dataset discovery

In [2]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Marketing spend:", marketing.shape)

Customers: (24123, 5)
Orders: (38366, 9)
Marketing spend: (53, 3)


In [3]:
print(customers.dtypes)
customers.head()



customer_id              str
first_order_date         str
acquisition_channel      str
region                   str
newsletter_optin       int64
dtype: object


,customer_id,first_order_date,acquisition_channel,region,newsletter_optin
0,C000594,2025-01-01,Organic/SEO,Auvergne-Rhône-Alpes,0
1,C000100,2025-01-01,Meta Ads,Nouvelle-Aquitaine,1
2,C000237,2025-01-01,Meta Ads,Île-de-France,0
3,C000030,2025-01-01,Meta Ads,Île-de-France,0
4,C000097,2025-01-01,Meta Ads,Île-de-France,1


In [4]:
print(orders.dtypes)
orders.head()

order_id                 str
customer_id              str
order_date               str
promo_code               str
n_items                int64
gross_product_eur    float64
discount_eur         float64
shipping_fee_eur     float64
main_category            str
dtype: object


,order_id,customer_id,order_date,promo_code,n_items,gross_product_eur,discount_eur,shipping_fee_eur,main_category
0,O0019640,C010318,2026-05-16,NaN,2,26.10,0.0,5.9,care_products
1,O0009942,C004872,2025-12-10,NaN,1,23.94,0.0,5.9,indoor_plants
2,O0036919,C024053,08/06/2026,NaN,2,49.60,0.0,0.0,indoor_plants
3,O0028444,C016562,2026-03-13,NaN,2,48.49,0.0,0.0,care_products
4,O0013903,C007018,2025-07-18,NaN,2,26.29,0.0,5.9,indoor_plants


In [5]:
print(marketing.dtypes)
marketing.head()

month            str
channel          str
spend_eur    float64
dtype: object


,month,channel,spend_eur
0,2025-01-01,Meta Ads,7200.0
1,2025-01-01,Google Shopping,4000.0
2,2025-01-01,Influencers,0.0
3,2025-02-01,Meta Ads,8260.0
4,2025-02-01,Google Shopping,4500.0


## 3. Convert dates columns

From the first rows of the Orders dataset we can conclude that the data contains ISO dates (`YYYY-MM-DD`) and day-first dates (`DD/MM/YYYY`). We convert both formats and use `NaT` for values that cannot be parsed for all the datasets.

### Customers dataset

In [6]:
# Keep the original row count for the cleaning log.
before = len(customers)

# Convert both YYYY-MM-DD and DD/MM/YYYY values to datetime.
original_dates = customers["first_order_date"]
customers["first_order_date"] = pd.to_datetime(
    original_dates,
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

# Compare with canonical ISO text to count reformatted and invalid values.
canonical_dates = customers["first_order_date"].dt.strftime("%Y-%m-%d")
affected_rows = original_dates.ne(canonical_dates).fillna(False).sum()

cleaning_records.append({
    "dataset": "customers",
    "column": "first_order_date",
    "rule": "Convert mixed date text to datetime",
    "rows_before": before,
    "rows_after": len(customers),
    "affected_rows": int(affected_rows),
    "action": "Converted valid dates; kept invalid values as NaT for review",
})

customers["first_order_date"].head()

0   2025-01-01
1   2025-01-01
2   2025-01-01
3   2025-01-01
4   2025-01-01
Name: first_order_date, dtype: datetime64[us]

### Orders dataset

In [7]:
# Keep the original row count for the cleaning log.
before = len(orders)
# Keep the original text so the log counts every changed representation.
original_dates = orders["order_date"].astype("string").str.strip()

# Convert both YYYY-MM-DD and DD/MM/YYYY values to datetime.
orders["order_date"] = pd.to_datetime(
    original_dates,
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

# Compare with canonical ISO text to count reformatted and invalid values.
canonical_dates = orders["order_date"].dt.strftime("%Y-%m-%d")
affected_rows = original_dates.ne(canonical_dates).fillna(False).sum()
cleaning_records.append({
    "dataset": "orders",
    "column": "order_date",
    "rule": "Convert mixed date text to datetime",
    "rows_before": before,
    "rows_after": len(orders),
    "affected_rows": int(affected_rows),
    "action": "Converted valid dates; kept invalid values as NaT for review",
})

orders["order_date"].head()

0   2026-05-16
1   2025-10-12
2   2026-06-08
3   2026-03-13
4   2025-07-18
Name: order_date, dtype: datetime64[us]

### Marketing dataset

In [8]:
# Keep the original row count for the cleaning log.
before = len(marketing)
# Keep the original text so the log counts every changed representation.
original_dates = marketing["month"].astype("string").str.strip()

# Convert both YYYY-MM-DD and DD/MM/YYYY values to datetime.
marketing["month"] = pd.to_datetime(
    original_dates,
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

# Compare with canonical ISO text to count reformatted and invalid values.
canonical_dates = marketing["month"].dt.strftime("%Y-%m-%d")
affected_rows = original_dates.ne(canonical_dates).fillna(False).sum()
cleaning_records.append({
    "dataset": "marketing_spend",
    "column": "month",
    "rule": "Convert mixed date text to datetime",
    "rows_before": before,
    "rows_after": len(marketing),
    "affected_rows": int(affected_rows),
    "action": "Converted valid dates; kept invalid values as NaT for review",
})

marketing["month"].head()

0   2025-01-01
1   2025-01-01
2   2025-01-01
3   2025-01-02
4   2025-01-02
Name: month, dtype: datetime64[us]

## 4. Check for duplicated rows

In [9]:
# Count complete duplicate rows in each current working dataset.
print("Nb of duplicated rows in Customers dataset:", customers.duplicated().sum())
print("Nb of duplicated rows in Orders dataset:",orders.duplicated().sum())
print("Nb of duplicated rows in marketing dataset:",marketing.duplicated().sum())

Nb of duplicated rows in Customers dataset: 0
Nb of duplicated rows in Orders dataset: 266
Nb of duplicated rows in marketing dataset: 0


In [10]:
# Check order_id uniqueness before removing complete duplicate rows.
order_id_counts = orders["order_id"].value_counts()
repeated_order_ids = orders[
    orders["order_id"].isin(order_id_counts[order_id_counts > 1].index)
].sort_values("order_id")

print("Number of order rows:", len(orders))
print("Number of unique order_id values:", orders["order_id"].nunique())
print("Rows with a repeated order_id:", len(repeated_order_ids))
display(repeated_order_ids)

cleaning_records.append({
    "dataset": "orders",
    "column": "order_id",
    "rule": "Check order_id uniqueness before duplicate removal",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": len(repeated_order_ids),
    "action": "Displayed all columns for repeated order_id values",
})

Number of order rows: 38366
Number of unique order_id values: 38100
Rows with a repeated order_id: 532


,order_id,customer_id,order_date,promo_code,n_items,gross_product_eur,discount_eur,shipping_fee_eur,main_category
9619,O0000158,C000089,2025-01-14,NaN,1,41.23,0.0,0.0,indoor_plants
15906,O0000158,C000089,2025-01-14,NaN,1,41.23,0.0,0.0,indoor_plants
34060,O0000170,C000094,2025-01-20,WELCOME30,2,62.33,18.7,0.0,indoor_plants
18940,O0000170,C000094,2025-01-20,WELCOME30,2,62.33,18.7,0.0,indoor_plants
11927,O0000438,C000230,2025-10-30,NaN,1,27.90,0.0,5.9,indoor_plants
...,...,...,...,...,...,...,...,...,...
29461,O0036853,C023988,2026-06-23,NaN,1,21.08,0.0,5.9,outdoor_plants
15848,O0036936,C024070,2026-07-06,NaN,1,15.23,0.0,5.9,indoor_plants
12058,O0036936,C024070,2026-07-06,NaN,1,15.23,0.0,5.9,indoor_plants
23119,O0036953,C024086,2026-06-28,NaN,1,46.25,0.0,0.0,outdoor_plants


## 5.Dealing with duplicated rows in the Orders dataset

In [11]:
# Keep one copy of each completely duplicated order row.
orders_before_duplicates = len(orders)
duplicate_rows = orders.duplicated(keep="first")# a True/False 
duplicated_order_rows = int(duplicate_rows.sum())#nb of true= nb of duplicated rows

# Remove only the repeated copies; the first occurrence is retained.
orders = orders.loc[~duplicate_rows]

cleaning_records.append({
    "dataset": "orders",
    "column": "all columns",
    "rule": "Remove complete duplicate order rows",
    "rows_before": orders_before_duplicates,
    "rows_after": len(orders),
    "affected_rows": duplicated_order_rows,
    "action": "Removed repeated copies; kept the first occurrence",
})

print(f"Removed {duplicated_order_rows} duplicated order rows.")
#verify
print("Nb of duplicated rows in Orders dataset:",orders.duplicated().sum())



Removed 266 duplicated order rows.
Nb of duplicated rows in Orders dataset: 0


## 6.internal or test accounts

In [12]:
# Search customer identifiers for common internal or test account markers.
account_markers = r"test|internal|dummy|example|sandbox|qa"
customer_id_text = customers["customer_id"].astype("string")
internal_test_customers = customers[
    customer_id_text.str.contains(account_markers, case=False, na=False)
]

print("Markers searched in customer_id:", account_markers)
print("Internal or test customer accounts found:", len(internal_test_customers))
display(internal_test_customers)

cleaning_records.append({
    "dataset": "customers",
    "column": "customer_id",
    "rule": "Check for internal or test account markers",
    "rows_before": len(customers),
    "rows_after": len(customers),
    "affected_rows": len(internal_test_customers),
    "action": "Displayed possible internal or test accounts; no rows removed automatically",
})

Markers searched in customer_id: test|internal|dummy|example|sandbox|qa
Internal or test customer accounts found: 0


,customer_id,first_order_date,acquisition_channel,region,newsletter_optin


## 7.Looking for missing values

In [13]:
print(customers.isna().sum())
print(orders.isna().sum())
print(marketing.isna().sum())

customer_id              0
first_order_date         0
acquisition_channel      0
region                 466
newsletter_optin         0
dtype: int64
order_id                 0
customer_id              0
order_date               0
promo_code           28872
n_items                  0
gross_product_eur        0
discount_eur             0
shipping_fee_eur         0
main_category            0
dtype: int64
month        0
channel      0
spend_eur    0
dtype: int64


So, we notice having missing values only for the region column in Customers' dataset and promo_code column in Orders' dataset (logic for orders without pomocode )

In [14]:
# Replace missing promo codes with NO_PROMO: missing means no promotion was used.
orders_before_promo = len(orders)
missing_promo = orders["promo_code"].isna()
orders["promo_code"] = orders["promo_code"].fillna("NO_PROMO")

cleaning_records.append({
    "dataset": "orders",
    "column": "promo_code",
    "rule": "Replace missing promo codes",
    "rows_before": orders_before_promo,
    "rows_after": len(orders),
    "affected_rows": int(missing_promo.sum()),
    "action": "Replaced missing values with NO_PROMO",
})

# Replace missing regions with Unknown: the actual region cannot be inferred safely.
customers_before_region = len(customers)
missing_region = customers["region"].isna()
customers["region"] = customers["region"].fillna("Unknown")

cleaning_records.append({
    "dataset": "customers",
    "column": "region",
    "rule": "Replace missing regions",
    "rows_before": customers_before_region,
    "rows_after": len(customers),
    "affected_rows": int(missing_region.sum()),
    "action": "Replaced missing values with Unknown",
})

## 8.Inconsistent labels


In [15]:
#Checking for Customers' dataset
print(customers['region'].value_counts())
print(customers['acquisition_channel'].value_counts())

#Checking for Orders' dataset
print(orders['main_category'].value_counts())

#Checking for Marketing's dataset
print(marketing['channel'].value_counts())


region
Île-de-France           7558
Other                   6195
Auvergne-Rhône-Alpes    3289
Nouvelle-Aquitaine      2417
Occitanie               2335
Hauts-de-France         1863
Unknown                  466
Name: count, dtype: int64
acquisition_channel
Meta Ads           9912
Organic/SEO        7451
Google Shopping    3973
Email/Referral     1787
Influencers        1000
Name: count, dtype: int64
main_category
indoor_plants     14792
outdoor_plants    12859
pots               6367
care_products      4082
Name: count, dtype: int64
channel
Meta Ads           18
Influencers        18
Google Shopping    17
Name: count, dtype: int64


According to the previous statistics, there is no inconsistent labels in our datasets.

## 9.Impossible values

Customers' dataset

In [16]:
print(customers['newsletter_optin'].value_counts())
print("Date minimale :", customers['first_order_date'].min())
print("Date maximale :", customers['first_order_date'].max())

newsletter_optin
0    16869
1     7254
Name: count, dtype: int64
Date minimale : 2025-01-01 00:00:00
Date maximale : 2026-12-06 00:00:00


In [17]:
# Find the earliest recorded order date for each customer.
earliest_order_dates = orders.groupby("customer_id")["order_date"].min()

customer_date_check = customers[["customer_id", "first_order_date"]].copy()
customer_date_check["earliest_order_date"] = customer_date_check["customer_id"].map(
    earliest_order_dates
)

# If an order predates the recorded first order, update the customer's date.
customer_date_updates = customer_date_check[
    customer_date_check["earliest_order_date"].notna()
    & (
        customer_date_check["earliest_order_date"]
        < customer_date_check["first_order_date"]
    )
]

updated_customer_ids = customer_date_updates["customer_id"]
customers.loc[
    customers["customer_id"].isin(updated_customer_ids),
    "first_order_date",
] = customers.loc[
    customers["customer_id"].isin(updated_customer_ids), "customer_id"
].map(earliest_order_dates)

print(
    "Customers whose first-order date was updated:",
    len(customer_date_updates),
)

cleaning_records.append({
    "dataset": "customers",
    "column": "first_order_date",
    "rule": "Set first-order date to the earliest recorded order date",
    "rows_before": len(customers),
    "rows_after": len(customers),
    "affected_rows": len(customer_date_updates),
    "action": "Updated customer first-order dates for customers with an earlier order",
})

# Recheck that every order date is on or after the corrected first-order date.
orders_with_customer_dates = orders.merge(
    customers[["customer_id", "first_order_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

invalid_order_dates = orders_with_customer_dates[
    orders_with_customer_dates["order_date"]
    < orders_with_customer_dates["first_order_date"]
]


cleaning_records.append({
    "dataset": "orders",
    "column": "order_date",
    "rule": "Validate order date against corrected customer first-order date",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": len(invalid_order_dates),
    "action": "Validation check after updating customer first-order dates",
})

Customers whose first-order date was updated: 1690


Orders' dataset

In [18]:
# Convert negative item counts to positive values .
negative_n_items = int((orders["n_items"] < 0).sum())
orders.loc[orders["n_items"] < 0, "n_items"] = orders.loc[
    orders["n_items"] < 0, "n_items"
].abs()
print("Negative n_items converted to positive values:", negative_n_items)

cleaning_records.append({
    "dataset": "orders",
    "column": "n_items",
    "rule": "Convert negative item counts to positive values",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": negative_n_items,
    "action": "Converted negative n_items values using absolute values",
})

Negative n_items converted to positive values: 1109


In [19]:
##date range
print("Date minimale :", customers['first_order_date'].min())
print("Date maximale :", customers['first_order_date'].max())

Date minimale : 2025-01-01 00:00:00
Date maximale : 2026-12-06 00:00:00


In [20]:
# A non-zero discount is valid only when the order has a promo code.
discount_without_promo = (
    orders["promo_code"].eq("NO_PROMO")
    & orders["discount_eur"].ne(0)
)
discount_without_promo_count = int(discount_without_promo.sum())

print(
    "Orders with a non-zero discount but no promo code:",
    discount_without_promo_count,
)

cleaning_records.append({
    "dataset": "orders",
    "column": "discount_eur",
    "rule": "A non-zero discount requires a promo code",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": discount_without_promo_count,
    "action": "Detected discount values without a promo code; no values changed automatically",
})

Orders with a non-zero discount but no promo code: 0


In [21]:
# Impute negative gross amounts with the median of a valid comparable group.
orders["_item_count_for_imputation"] = orders["n_items"].abs()
orders["_free_shipping_for_imputation"] = orders["shipping_fee_eur"].eq(0)
negative_gross_product = int((orders["gross_product_eur"] < 0).sum())
valid_gross_orders = orders[
    orders["gross_product_eur"].ge(0)
    & orders["shipping_fee_eur"].notna()
]
gross_medians = (
    valid_gross_orders
    .groupby([
        "main_category",
        "_item_count_for_imputation",
        "_free_shipping_for_imputation",
    ])["gross_product_eur"]
    .median()
)
gross_group_keys = pd.MultiIndex.from_frame(
    orders[
        [
            "main_category",
            "_item_count_for_imputation",
            "_free_shipping_for_imputation",
        ]
    ]
)
imputed_gross_values = pd.Series(
    gross_medians.reindex(gross_group_keys).to_numpy(),
    index=orders.index,
)
negative_gross_mask = orders["gross_product_eur"] < 0
missing_gross_medians = int(
    (negative_gross_mask & imputed_gross_values.isna()).sum()
)
if missing_gross_medians > 0:
    raise ValueError(
        "No valid median group is available for "
        f"{missing_gross_medians} negative gross amount(s)."
    )
orders.loc[negative_gross_mask, "gross_product_eur"] = imputed_gross_values[
    negative_gross_mask
]
orders.drop(
    columns=[
        "_item_count_for_imputation",
        "_free_shipping_for_imputation",
    ],
    inplace=True,
)
print("Negative values in gross_product_eur imputed:", negative_gross_product)
cleaning_records.append({
    "dataset": "orders",
    "column": "gross_product_eur",
    "rule": "Impute negative gross amounts using the median of valid orders with the same shipping status, main category, and item count",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": negative_gross_product,
    "action": "Replaced negative values with positive group medians; free shipping means fee 0 and paid shipping means fee non-zero",
})

negative_discount = int((orders["discount_eur"] < 0).sum())
orders.loc[orders["discount_eur"] < 0, "discount_eur"] = orders.loc[
    orders["discount_eur"] < 0, "discount_eur"
].abs()
print("Negative discounts converted to positive values:", negative_discount)
cleaning_records.append({
    "dataset": "orders",
    "column": "discount_eur",
    "rule": "Convert negative discount values to positive values",
    "rows_before": len(orders),
    "affected_rows": negative_discount,
    "action": "Converted negative discount values using absolute values",
})

negative_spend = int((marketing["spend_eur"] < 0).sum())
print("Negative values in spend_eur:", negative_spend)
cleaning_records.append({
    "dataset": "marketing",
    "column": "spend_eur",
    "rule": "Money values must be non-negative",
    "rows_before": len(marketing),
    "rows_after": len(marketing),
    "affected_rows": negative_spend,
    "action": "Detected negative money values; no values changed automatically",
})

Negative values in gross_product_eur imputed: 1109
Negative discounts converted to positive values: 270
Negative values in spend_eur: 0


C:\Users\user\AppData\Local\Temp\ipykernel_29580\3312761153.py:43: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders.drop(


In [22]:
# Calculate the order amount after applying the discount.
net_order_amount = orders["gross_product_eur"] - orders["discount_eur"]

# Orders above 40 EUR must have free shipping.
free_shipping_rule = (net_order_amount > 40) & (orders["shipping_fee_eur"] != 0)
free_shipping_corrections = int(free_shipping_rule.sum())
orders.loc[free_shipping_rule, "shipping_fee_eur"] = 0.0

cleaning_records.append({
    "dataset": "orders",
    "column": "shipping_fee_eur",
    "rule": "Free shipping for net order amounts above 40 EUR",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": free_shipping_corrections,
    "action": "Set shipping_fee_eur to 0.0",
})

# Orders below 40 EUR must have the standard 5.90 EUR shipping fee.
paid_shipping_rule = (net_order_amount < 40) & (orders["shipping_fee_eur"] != 5.9)
paid_shipping_corrections = int(paid_shipping_rule.sum())
orders.loc[paid_shipping_rule, "shipping_fee_eur"] = 5.9

cleaning_records.append({
    "dataset": "orders",
    "column": "shipping_fee_eur",
    "rule": "Standard shipping for net order amounts below 40 EUR",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": paid_shipping_corrections,
    "action": "Set shipping_fee_eur to 5.90",
})

print("Free-shipping corrections:", free_shipping_corrections)
print("Paid-shipping corrections:", paid_shipping_corrections)


Free-shipping corrections: 4
Paid-shipping corrections: 651


## 9. Save this cleaning phase


In [23]:
cleaning_log = pd.DataFrame(cleaning_records)
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False)

customers.to_csv(OUTPUT_DIR / "customers_cleaned.csv", index=False)
orders.to_csv(OUTPUT_DIR / "orders_cleaned.csv", index=False)
marketing.to_csv(OUTPUT_DIR / "marketing_cleaned.csv", index=False)

print("Saved the typed datasets and cleaning log.")

Saved the typed datasets and cleaning log.
